# Choosing the third channel for the 3-channel CXR pipeline

Fixed setup (not under test here):

- **ch0 = raw** grayscale CXR (`/255`)
- **ch1 = CLAHE(clip=2.0, tile=8x8)** -- the pinned mild CLAHE
- **ch2 = candidate** -- the one degree of freedom we are evaluating

**Read this before trusting any number below.** All three channels are the *same
anatomy*, so ch1 and ch2 will always be **highly correlated** (a bright lung is
bright in every channel). That is expected and is **not** a defect.

Correlation is used here only as a **redundancy diagnostic** -- "does ch2 add
anything ch1 doesn't?" -- *not* as a score to maximize. A very *low* correlation is
**not automatically better**: an edge/Sobel map or an over-amplified CLAHE has low
correlation precisely because it is dominated by noise and artifacts, which is worse,
not better. The right third channel **balances**:

1. **redundancy** -- some unique variance vs ch1 (not ~0, not ~1),
2. **radiograph-likeness** -- still looks like a chest X-ray, not a noise field,
3. **artifact amplification** -- no blown-out tile seams / haloing,
4. **downstream validation** -- the only true tie-breaker (done elsewhere, not here).

This notebook **does not train anything**. It shortlists 2-3 reasonable candidates
for you to take to a real validation run.


## 1. Configuration

Edit `CANDIDATES`, `N_SAMPLE`, and the preview image here.

In [ ]:
import os
os.environ['CURRENT_NOTEBOOK_NAME'] = 'choose_third_channel'
# --- Config ------------------------------------------------------------------
N_SAMPLE   = 1000     # images for the aggregate statistics
PREVIEW_N  = 1        # how many single-image visual panels to print first
SAMPLE_SEED = 42      # reproducible aggregate sample
PREVIEW_SEED = 7      # reproducible preview image pick
SAVE = True           # write summary + per-image CSVs next to this notebook

# Fixed reference channel (ch1). Not a candidate -- every candidate is compared to it.
CH1_CLIP, CH1_TILE = 2.0, (8, 8)

# Candidate third channels (ch2). Name -> spec. Edit freely.
#   ("clahe", clip, tile) | ("histeq",) | ("sobel",) | ("raw",)
CANDIDATE_SPECS = [
    ("clahe_3.0_16x16", ("clahe", 3.0, (16, 16))),   # main candidate
    ("clahe_4.0_16x16", ("clahe", 4.0, (16, 16))),   # main candidate
    ("clahe_5.0_16x16", ("clahe", 5.0, (16, 16))),   # main candidate
    ("clahe_5.0_24x24", ("clahe", 5.0, (24, 24))),   # main candidate
    ("histeq",          ("histeq",)),                 # main candidate
    ("sobel",           ("sobel",)),                  # main candidate (expect LOW corr = mostly noise)
    ("raw_dup",         ("raw",)),                     # baseline: ch2 == ch0 (sanity reference)
]


In [ ]:
# --- Setup: imports, repo root, channel builders, sample loading -------------
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count

cv2.setNumThreads(1)  # parallelize across images, not within one

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.reproducibility import seed_everything
from src.dataloader.utils import resolve_preferred_image_path, resolve_image_path
seed_everything(SAMPLE_SEED)

TRAIN_DF_PATH = REPO_ROOT / "data" / "data-camchex" / "03_mimic_train.csv"
NUM_WORKERS = max(1, int(cpu_count() * 0.7))


def read_gray(path):
    """Grayscale uint8 read using project path-resolution logic."""
    resolved = resolve_preferred_image_path(path)
    candidates = [resolved]
    if "_resized_1024.jpg" in resolved:
        candidates.append(resolved.replace("_resized_1024.jpg", ".jpg"))
    else:
        candidates.append(resolve_image_path(path))
    for cand in candidates:
        img = cv2.imread(cand, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            return img
    return None


# --- channel builders: uint8 gray -> float32 [0,1] ---------------------------
def ch_raw(g):
    return g.astype(np.float32) / 255.0


def ch_clahe(g, clip, tile):
    return cv2.createCLAHE(clipLimit=clip, tileGridSize=tuple(tile)).apply(g).astype(np.float32) / 255.0


def ch_histeq(g):
    return cv2.equalizeHist(g).astype(np.float32) / 255.0


def ch_sobel(g):
    gx = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)
    m = np.sqrt(gx * gx + gy * gy)
    lo, hi = float(m.min()), float(m.max())
    return ((m - lo) / (hi - lo)).astype(np.float32) if hi > lo else np.zeros_like(m, dtype=np.float32)


def build_channel(spec, g):
    kind = spec[0]
    if kind == "raw":
        return ch_raw(g)
    if kind == "clahe":
        return ch_clahe(g, spec[1], spec[2])
    if kind == "histeq":
        return ch_histeq(g)
    if kind == "sobel":
        return ch_sobel(g)
    raise ValueError(f"unknown channel spec {spec!r}")


def ch1_of(g):
    return ch_clahe(g, CH1_CLIP, CH1_TILE)


# --- load the training split and draw the aggregate sample -------------------
df = pd.read_csv(TRAIN_DF_PATH)
if "split" in df.columns:
    df = df[df["split"] == "train"]
sample_df = df.sample(n=min(N_SAMPLE, len(df)), random_state=SAMPLE_SEED).reset_index(drop=True)
print(f"train rows: {len(df):,} | aggregate sample: {len(sample_df):,} | workers: {NUM_WORKERS}")
print("candidates:", [n for n, _ in CANDIDATE_SPECS])


In [ ]:
# --- per-image metrics for one candidate vs ch1 ------------------------------
def candidate_metrics(ch1, ch2):
    a = ch1.reshape(-1).astype(np.float64)
    b = ch2.reshape(-1).astype(np.float64)
    # Pearson corr; guard the degenerate (constant channel) case.
    sa, sb = a.std(), b.std()
    corr = float(np.corrcoef(a, b)[0, 1]) if sa > 1e-9 and sb > 1e-9 else np.nan
    p1, p50, p99 = np.percentile(b, [1, 50, 99])
    return {
        "corr": corr,                              # redundancy diagnostic (expected high)
        "unique_var": 1.0 - corr ** 2 if corr == corr else np.nan,  # 1 - r^2
        "mean_abs_diff": float(np.mean(np.abs(b - a))),
        "ch2_mean": float(b.mean()),
        "ch2_std": float(b.std()),
        "ch2_p1": float(p1),
        "ch2_p50": float(p50),
        "ch2_p99": float(p99),
    }


## 2. Single-image visual check (run first)

Before any averaging, look at **one** image. Numbers can't tell you whether a
channel is full of tile seams or looks like noise -- your eye can. For each
candidate, the row shows: **raw | ch1 (fixed CLAHE) | candidate ch2 |
|ch2 - ch1| difference map | RGB composite (R=raw, G=ch1, B=ch2)**.

The difference map is where ch2 *disagrees* with ch1: a useful channel shows
structured anatomical differences; a bad one shows uniform noise or blown-out
seams. The printed table is the same metrics, for this single image only.


In [ ]:
# --- pick PREVIEW_N readable studies and show the panels ---------------------
preview = []
for _p in sample_df["path"].sample(frac=1.0, random_state=PREVIEW_SEED):
    if len(preview) >= PREVIEW_N:
        break
    _g = read_gray(_p)
    if _g is not None:
        preview.append((_p, _g))

for path, gray in preview:
    ch1 = ch1_of(gray)
    rows = []
    ncols = 5
    fig, axes = plt.subplots(len(CANDIDATE_SPECS), ncols,
                             figsize=(2.6 * ncols, 2.5 * len(CANDIDATE_SPECS)), squeeze=False)
    col_titles = ["raw (ch0)", f"ch1 CLAHE {CH1_CLIP}/{CH1_TILE[0]}x{CH1_TILE[1]}",
                  "candidate ch2", "|ch2 - ch1|", "RGB (raw,ch1,ch2)"]
    raw = ch_raw(gray)
    for r, (name, spec) in enumerate(CANDIDATE_SPECS):
        ch2 = build_channel(spec, gray)
        diff = np.abs(ch2 - ch1)
        rgb = np.stack([raw, ch1, ch2], axis=-1)
        panels = [raw, ch1, ch2, diff, rgb]
        for c, img in enumerate(panels):
            ax = axes[r][c]
            ax.imshow(img, cmap=(None if c == 4 else "gray"), vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=9)
        axes[r][0].set_ylabel(name, fontsize=8)
        m = candidate_metrics(ch1, ch2)
        m = {"candidate": name, **m}
        rows.append(m)
    fig.suptitle(f"Single-image check -- {Path(str(path)).stem[:28]}", fontsize=11)
    fig.tight_layout()
    plt.show()

    one = pd.DataFrame(rows).set_index("candidate")
    print(f"single-image metrics ({Path(str(path)).stem[:28]}):")
    display(one.round(4))


## 3. Aggregate over `N_SAMPLE` images

Now average the per-image metrics over the sample to see how each candidate
behaves *on average* (a single image can be unusual). Metrics are computed
per-image and then averaged; `corr_std` shows how much the redundancy varies
across images.


In [ ]:
# --- threaded aggregate: per-image metrics, then average ---------------------
from IPython.display import display

CAND_NAMES = [n for n, _ in CANDIDATE_SPECS]


def metrics_for_path(path):
    g = read_gray(path)
    if g is None:
        return None
    ch1 = ch1_of(g)
    out = {}
    for name, spec in CANDIDATE_SPECS:
        out[name] = candidate_metrics(ch1, build_channel(spec, g))
    return out


per_image_records = []  # long form: one row per (image, candidate)
skipped = 0
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futures = {ex.submit(metrics_for_path, p): i for i, p in enumerate(sample_df["path"])}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="images"):
        res = fut.result()
        if res is None:
            skipped += 1
            continue
        idx = futures[fut]
        for name in CAND_NAMES:
            per_image_records.append({"image_idx": idx, "candidate": name, **res[name]})

per_image_df = pd.DataFrame(per_image_records)
print(f"processed {per_image_df['image_idx'].nunique()} images | skipped {skipped}")

# Aggregate: mean of every metric per candidate, plus the spread of corr.
metric_cols = ["corr", "unique_var", "mean_abs_diff", "ch2_mean", "ch2_std",
               "ch2_p1", "ch2_p50", "ch2_p99"]
summary_df = per_image_df.groupby("candidate")[metric_cols].mean()
summary_df["corr_std"] = per_image_df.groupby("candidate")["corr"].std()
# Preserve the order we defined the candidates in.
summary_df = summary_df.loc[CAND_NAMES]


In [ ]:
# --- summary table -----------------------------------------------------------
display_cols = ["corr", "corr_std", "unique_var", "mean_abs_diff",
                "ch2_mean", "ch2_std", "ch2_p1", "ch2_p50", "ch2_p99"]
print("Aggregate over the sample (averaged per-image metrics). ch1 = CLAHE "
      f"{CH1_CLIP}/{CH1_TILE[0]}x{CH1_TILE[1]}.")
print("Read 'corr' as redundancy (high is expected), NOT as a score to minimize.\n")
display(summary_df[display_cols].round(4))

if SAVE:
    out_summary = Path.cwd() / "third_channel_summary.csv"
    out_perimg = Path.cwd() / "third_channel_per_image.csv"
    summary_df.to_csv(out_summary)
    per_image_df.to_csv(out_perimg, index=False)
    print(f"\nsaved: {out_summary.name}  ({len(summary_df)} rows)")
    print(f"saved: {out_perimg.name}  ({len(per_image_df):,} rows)")


## 4. How to read this table (shortlist, don't auto-pick)

There is **no single 'best' column**. Use the table to *shortlist 2-3*, then let a
real validation run decide. Rules of thumb:

- **`corr` / `unique_var`** -- redundancy. You want a candidate that is *not*
  near-identical to ch1 (corr ~0.99, unique_var ~0) but also *not* near-zero
  correlation. Mid-range unique variance means ch2 adds real, structured info.
- **`sobel` is the cautionary example** -- it will show the *lowest* corr / highest
  `mean_abs_diff`, but that's because it's a sparse edge map (mostly near-zero,
  dominated by noise/artifacts). Low correlation here is **bad**, not good. This is
  exactly why correlation alone must never pick the winner.
- **`raw_dup`** -- the baseline. ch2 == ch0, so it tells you what "a second copy of
  raw" scores; a useful candidate should beat it on uniqueness *while staying dense*.
- **`ch2_std` and the percentiles** -- a healthy dense channel has std ~0.25-0.31 and
  a wide p1->p99 spread. Collapsing std (like the edge channels, ~0.02) means the
  channel will normalize poorly. Very high `ch2_p99` with very low `ch2_p1` plus
  visible seams in the single-image check = over-amplified CLAHE (artifacts).
- **The single-image panels (section 2)** are the artifact veto: if a candidate looks
  blown-out or noisy there, drop it no matter how its numbers look.

**Suggested shortlist heuristic:** keep the 2-3 *dense* CLAHE candidates whose
`unique_var` is meaningfully above `raw_dup` but that still look like a radiograph
in the panels -- then take those to a downstream validation run to break the tie.
